# Working with the Labour Force Survey (in R)
## UK Data Service datasets

In this notebook, we aim to write functionality to simplify locating, looking up, renaming and recoding variables in [UKDS datasets](https://ukdataservice.ac.uk/).

UKDS data dictionaries in the **Understanding Society** and **Labour Force Survey** datasets are buried in `rtf` files in nested archives. Unzipping these manually can be effort intensive, recoding variables even more so. We can use R or [Python](https://github.com/tros01/econometrics_projects/blob/main/ukds_data/lfs-py.ipynb) to automate parts of these processes. 

In this notebook, we write functions that unzip, relocate and rename files should we wish to simplify access to individual datasets archived in the original zip files.

We then define functions that extract the data dictionary information from an `rtf` file stored in a nested zip file and store them in useful dictionaries we can iterate through when we need to look up or recode variables.

The code has been tested on select LFS survey archives. All functions are loop-ready should you wish to process multiple datasets at a time.

The data are available directly from the [UK Data service](https://ukdataservice.ac.uk/) and require individual registration.

### Custom functions in this notebook

| | R |
|---|---|
| List zip archive contents | `itemise_archive` |
| Open or extract a file from an archive | `extract_file`, `copy_archived_file` |
| Read the rtf data dictionary as text | `read_data_dictionary` |
| Variable -> list(column position, label) | `variable_dictionary` |
| Variable -> list(code -> label) | `code_dictionary` |
| Read a `.tab` file with cleaned names | `read_tab` |

In [4]:
# Libraries
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [5]:
# LFS 2024-2026 archives folder
data_root <- r"(F:\OneDrive\coding_files\data\lfs_2024-2026)"

print("Did we get the path right?")
dir.exists(data_root)

[1] "Did we get the path right?"


[1] TRUE

## Archives

UKDS datasets come in zip files. We list the required files first.

In [103]:
# List all tab archives
lfs_files <- list.files(data_root, pattern = ".+tab.+zip$", full.names = TRUE, recursive = TRUE)

cat("File count:")
print(length(lfs_files))

cat("\nAvailable tab-delimited datasets:")
lfs_files

File count:

[1] 22

Available tab-delimited datasets:

[1] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9264tab_E4EB2F2F5DC05C27D6DC9DA4527F4E41B1CE7090AD7E6C35FCBB0F9EBC11D35D_V1.zip"
 [2] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9270tab_B4A81ED99DAEB16984A7A74E1EB3D3E043C0ECB23E739F1030C1B9F2D3509FDD_V1.zip"
 [3] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9271tab_9CF3D2FC4DC44C17A8FC36FC16E80665224ECCB03C396FDBCE2EA11F6986241D_V1.zip"
 [4] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9272tab_1EAF2F77A3F94166741B7AF117053DA3DB1D3B1D019C6CC6FA4062C54E92BCE4_V1.zip"
 [5] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9297tab_D42E65C2B512DF771A004B37D76F4E31F591A47B63069B0CBF199A8A6CCC350A_V1.zip"
 [6] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9303tab_0AA54F15FACE9935D2A646A6E0B6862CF9D955C13B92A8326CC4ABA9463BFDF2_V1.zip"
 [7] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9323tab_E0073C655655038B6C7002C5BB0B9EA162FCE2D25CB1DE1EB083372246D5B2CB_V1.zip"
 [8] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9328tab_9A366D1B4491755BF6EFACDDA0593A3008E002748F06980CB701E53D03428AD9_V1.zip"
 [9] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9349tab_94B365E1EEB4127DCD87ADBEDC3FB03A8F8A3B0BE150161015DBFB8068A55DC4_V1.zip"
[10] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9353tab_56B9A6D0E300CA30EDC33F03E38AAC30B663C1EEF09AA09D8774159C0D2F8E62_V1.zip"
[11] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9388tab_01E21006332F72041FC32A0ED09C55B145610D51C46EE75C90004E1D74D1DBB5_V1.zip"
[12] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9403tab_BF36447E6BB364DC775ED223F6647BCBC9844F3ED9075BA16CA88FA3F78B177D_V1.zip"
[13] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9433tab_5EB5367E0A5ABCE311BDA3027CA894E9DB16941E151F18423D952EDA544E5071_V1.zip"
[14] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9445tab_8F11812797E3819153A71F9CB4A9BE6EB752658141FC79F30A6EA042F604724C_V1.zip"
[15] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9480tab_48636C01E631AB854FECF27E034A4041D9BB187454074A0F85D7C8510ACB1DE3_V1.zip"
[16] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9495tab_E93B9916F201DE04E2EDFD0896D6F32ABA5B4140743F42B24CF32EB606D7CF81_V1.zip"
[17] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9522tab_E2527DB76528649F9E2E8EE548BE5D0059AAF138F61ABCA48B7E670628D9D51E_V1.zip"
[18] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9532tab_1D7EB818A5D153C3BA3A5BD81453FDFFBA3F64CB26B835837BBE28AF93168070_V1.zip"
[19] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9569tab_08A17203E7FE9BA6903559828F61C60D093535C88548EDD6E97A721F6D031DE5_V1.zip"
[20] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9577tab_2205507874957A909DE707C6B5A583BB9FB8CCEE74AFB3E5C3A22350F95C4E84_V1.zip"
[21] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9607tab_5472DC661B9375F09CD4D6FDE7413FDEC43E68C1D5D5F1C6EDAB7BB56DEEDB04_V1.zip"
[22] "F:\\OneDrive\\coding_files\\data\\lfs_2024-2026/9622tab_0636A394EA987FB919EDC8ADD680F6F781D517882372401BF6D161BA3B9E7DB5_V1.zip"

We test access to the archived files on the first archive in the list. Note that the block is loop-ready. We can wrap it in a `for` loop and iterate through the entire list.

In [7]:
itemise_archive <- function(zip_path) {
    return(unzip(zip_path, list = TRUE))
}

In [106]:
# Itemise the contents of each archive
zip_inventory <- list()
for (i in seq_along(lfs_files)) {
    zip_inventory[[i]] <- itemise_archive(lfs_files[[i]])[["Name"]]
}    

cat("File count:\n")
print(length(zip_inventory))

cat("\nContents of an archive:")
zip_inventory[1]

File count:
[1] 22

Contents of an archive:

[[1]]
 [1] "UKDA-9264-tab/9264_file_information.rtf"                                                
 [2] "UKDA-9264-tab/mrdoc/ukda_data_dictionaries.zip"                                         
 [3] "UKDA-9264-tab/mrdoc/pdf/biennial_lfs_aps_reweighting_policy.pdf"                        
 [4] "UKDA-9264-tab/mrdoc/pdf/coronavirus_and_its_impact_on_the_labour_force_survey.pdf"      
 [5] "UKDA-9264-tab/mrdoc/pdf/labour_force_survey_newsletter_april_2020.pdf"                  
 [6] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol10_analysis_of_data_collected_by_lfs_2024.pdf"
 [7] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol11_longitudinalguide2022.pdf"                 
 [8] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol1_background_2024.pdf"                        
 [9] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol2_questionnaire_2022jscombined.pdf"           
[10] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol3_variabledetails2022js.pdf"                  
[11] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol3_variable_details_1992-2002.pdf"             
[12] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol4a_educationsupplement2022.pdf"               
[13] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol4_derived_variables_2020.pdf"                 
[14] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol5_classifications_2024.pdf"                   
[15] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol6_aps_user_guide_2024.pdf"                    
[16] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol7_variable_details_1979_1991.pdf"             
[17] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol8_household_2008.pdf"                         
[18] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol9a_eurostat_adhoc_modules_2008-2019.pdf"      
[19] "UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol9_eurostat_2010-2017.pdf"                     
[20] "UKDA-9264-tab/tab/lfsp_jm24_eul_pwt24.tab"                                              
[21] "UKDA-9264-tab/read9264.htm"                                                             
[22] "UKDA-9264-tab/mrdoc/UKDA/UKDA_Study_9264_Information.htm"

## Extracting files

We can extract a file and move it to a more convenient location under a more convenient name.

In [9]:
src <- r"(F:\temp\src\tst.txt)"
tgt <- r"(F:\temp\tgt)"

copy_file <- function(source_file_path, target_folder_path) {
    if (!file.exists((source_file_path))) {
        warning("The source file does not exist.")
    }
    if (!dir.exists(target_folder_path)) {
        dir.create(target_folder_path)
    }

    file_name <- basename(source_file_path)

    file.copy(
        from=source_file_path, to=target_folder_path, 
        overwrite = TRUE, recursive = FALSE, 
        copy.mode = TRUE
    )
    print(sprintf("'%s' copied to '%s'.", file_name, target_folder_path))
}

copy_file(src, tgt)

[1] "'tst.txt' copied to 'F:\\temp\\tgt'."


In [ ]:
i <- 1
archive_path <- lfs_files[[i]]
archived_file_name <- Filter(\(f) str_detect(f, "file_information"), itemise_archive(archive_path)[["Name"]])[[1]]
tgt_folder <- r"(F:\temp\tgt)"

copy_archived_file <- function(archive_path, archived_file_name, target_folder_path) {
    if (!file.exists((archive_path))) {
        warning("The source file does not exist.")
    }
    if (!dir.exists(target_folder_path)) {
        dir.create(target_folder_path)
    }

    file_name <- basename(archived_file_name)

    tryCatch(
        unzip(zipfile = archive_path, files = archived_file_name, exdir = target_folder_path, junkpaths = TRUE),
        warning = \(w) stop(sprintf("Could not extract '%s': %s", archived_file_name, conditionMessage(w)), call. = FALSE)
    )

    print(sprintf("'%s' extracted into '%s'.", file_name, target_folder_path))
}

copy_archived_file(archive_path, archived_file_name, tgt_folder)

[1] "'9264_file_information.rtf' extracted into 'F:\\temp\\tgt'."


## Data dictionaries

Dataset specific documentation is stored in `rtf` files in nested archives. We need to read up the `rtf` file two layers deep. We get a dictionary exactly as we see it when we open the `rtf` file in MS Word.

In [ ]:
read_rtf <- function(rtf_file) {
    # Avoid `striprtf::read_rtf()` unless necessary
    # Credit to Claude Code
    text <- readChar(rtf_file, file.size(rtf_file), useBytes = TRUE) |>
        # Convert a character vector between encoding
        iconv(from = "CP1252", to = "UTF-8") |>
        str_remove_all(r"(\{\\(fonttbl|colortbl)(\{[^{}]*\}|[^{}])*\})") |>
        str_replace_all(r"(\\par\b ?)", "\n") |>
        str_remove_all(r"(\\[a-z]+-?\d* ?|\\ |[{}])")
    return(text)
}

In [109]:
read_data_dictionary <- function(zip_file) {
    tmp <- tempfile()
    on.exit(unlink(tmp, recursive = TRUE))

    nested_zip_name <- str_subset(itemise_archive(zip_file)[["Name"]], "data_dictionaries")[[1]]
    nested_zip <- unzip(zip_file, files = nested_zip_name, exdir = tmp, junkpaths = TRUE)

    rtf_name <- str_subset(itemise_archive(nested_zip)[["Name"]], "rtf$")[[1]]
    rtf_file <- unzip(nested_zip, files = rtf_name, exdir = tmp, junkpaths = TRUE)

    return(read_rtf(rtf_file))
}

# Extract a data dictionary
i <- 1

data_dictionary_doc <- read_data_dictionary(lfs_files[[i]])

cat("Character count:\n")
print(nchar(data_dictionary_doc))

cat("\nSample from the data dictionary document:\n")
cat(str_split_1(data_dictionary_doc, "\n")[1:50], sep = "\n")

Character count:
[1] 758977

Sample from the data dictionary document:

UK Data Archive Data Dictionary

File-level information:

File Name = 		lfsp_jm24_eul_pwt24
Number of variables = 	801
Number of cases = 	50783


Variable-level information:

Pos. = 1	Variable = ACCDNT	Variable label = Accident resulting in injury at work
This variable is  numeric, the SPSS measurement level is NOMINAL
SPSS user missing values = -8.0 thru -9.0
	Value label information for ACCDNT
	Value = -9.0	Label = Does not apply
	Value = -8.0	Label = No answer
	Value = 1.0	Label = Yes
	Value = 2.0	Label = No

Pos. = 2	Variable = ACCKIND	Variable label = Type of Accident
This variable is  numeric, the SPSS measurement level is NOMINAL
SPSS user missing values = -8.0 thru -9.0
	Value label information for ACCKIND
	Value = -9.0	Label = Does not apply
	Value = -8.0	Label = No answer
	Value = 1.0	Label = Contact with moving machinery
	Value = 2.0	Label = Hit by moving, flying or falling object
	Value = 3.0	Label = Hi

We are actually interested in automating selection, renaming and access. A dicitonary mapping variable names onto the respective column position and variable label is therefore a useful idea.

In [130]:
variable_dictionary <- function(data_dictionary_doc) {
    var_pat <- r"(Pos\. = (\d+)\tVariable = (\w+)\tVariable label = (.+)\n)"
    match <- str_match_all(data_dictionary_doc, var_pat)[[1]]
    var_list <- setNames(
        Map(\(pos, lab) list(pos = pos, lab = lab), match[, 2], match[, 4]),
        janitor::make_clean_names(match[, 3])
    )

    return(var_list)
}

var_dict <- variable_dictionary(data_dictionary_doc)

cat("Item count:\n")
print(length(var_dict))

cat("\nAny empty values?\n")
empties1 <- Filter(\(n) if (length(var_dict[[n]]) == 0) n, names(var_dict))
if (length(empties1) == 0) {
    print("No empty values.")
} else {
    print(empties1)
}

cat("\nSample from the variable dictionary:\n\n")
print(var_dict |> head(10))

Item count:
[1] 801

Any empty values?
[1] "No empty values."

Sample from the variable dictionary:

$accdnt
$accdnt$pos
[1] "1"

$accdnt$lab
[1] "Accident resulting in injury at work"


$acckind
$acckind$pos
[1] "2"

$acckind$lab
[1] "Type of Accident"


$accurh1
$accurh1$pos
[1] "3"

$accurh1$lab
[1] "injuries"


$accurh2
$accurh2$pos
[1] "4"

$accurh2$lab
[1] "injuries"


$accurh3
$accurh3$pos
[1] "5"

$accurh3$lab
[1] "injuries"


$accurh4
$accurh4$pos
[1] "6"

$accurh4$lab
[1] "injuries"


$acthr
$acthr$pos
[1] "7"

$acthr$lab
[1] "Actual hours excluding overtime"


$acthr2
$acthr2$pos
[1] "8"

$acthr2$lab
[1] "2nd job actual hours including overtime"


$actpot
$actpot$pos
[1] "9"

$actpot$lab
[1] "Actual hours of paid overtime"


$actuot
$actuot$pos
[1] "10"

$actuot$lab
[1] "Actual hours of unpaid overtime"




It is also a good idea to construct another dictionary of dictionaries mapping variable names onto codes and values. This will come handy when we want to search an existing schema or recode it.

In [135]:
code_dictionary <- function(data_dictionary_doc) {
    code_pat <- r"(Pos\. = \d+\tVariable = (\w+)\t.*\n|\tValue = (\S+)\s*\tLabel = (.*)\n)"
    match <- str_match_all(data_dictionary_doc, code_pat)[[1]]
    
    var_indices <- !is.na(match[, 2])
    match[var_indices, 2] <- janitor::make_clean_names(match[var_indices, 2])
    match[, 2] <- zoo::na.locf(match[, 2])
    match <- match[!is.na(match[, 4]),] # The problem seems to be here

    var_names <- unique(match[, 2])
    code_list <- setNames(
        lapply(var_names, \(v) setNames(match[match[, 2] == v, 4], match[match[, 2] == v, 3])),
        var_names
    )
    
    return(code_list)
}

code_dict <- code_dictionary(data_dictionary_doc)

cat("Item count:\n")
print(length(code_dict))

cat("\nAny empty values?\n")
empties2 <- Filter(\(n) if (length(code_dict[[n]]) == 0) n, names(code_dict))
if (length(empties2) == 0) {
    print("No empty values.")
} else {
    print(empties2)
}

cat("\nSample from the code dictionary:\n\n")
print(code_dict |> head(5))

Item count:
[1] 799

Any empty values?
[1] "No empty values."

Sample from the code dictionary:

$accdnt
            -9.0             -8.0              1.0              2.0 
"Does not apply"      "No answer"            "Yes"             "No" 

$acckind
                                     -9.0 
                         "Does not apply" 
                                     -8.0 
                              "No answer" 
                                      1.0 
          "Contact with moving machinery" 
                                      2.0 
"Hit by moving, flying or falling object" 
                                      3.0 
                "Hit by a moving vehicle" 
                                      4.0 
      "Hit something fixed or stationary" 
                                      5.0 
  "Injured handling, lifting or carrying" 
                                      6.0 
 "Slipped, tripped or fell on same level" 
                                      7.0 
                

In [133]:
cat("Are there any discrepancies between variable names?\n")
d1 <- setdiff(
    var_dict |> names(),
    code_dict |> names()
)

if (length(d1) == 0) {
    print("No discrepancies.")
} else {
    print(d1)
}

Are there any discrepancies between variable names?
[1] "pwt24"  "refdte"


In [134]:
d2 <- setdiff(
    code_dict |> names(),
    var_dict |> names()
)

if (length(d2) == 0) {
    print("No discrepancies.")
} else {
    print(d2)
}

[1] "No discrepancies."


Which matches our Python solution.

## Datasets

Let's test access to an archived dataset.

In [30]:
extract_file <- function(zip_path, file_name) {
    f <- unz(zip_path, file_name)
    return(f)
}

read_tab <- function(file, nrows=Inf, cols=NULL) {
    df <- read_delim(
        file,
        delim = "\t",
        n_max = nrows,
        col_select = all_of(cols),
        show_col_types = FALSE
    )
    return(janitor::clean_names(df))
}

In [31]:
# Test df extract
i <- 1
tab_file_name <- Filter(\(f) str_detect(f, "tab$"), itemise_archive(lfs_files[[i]])[["Name"]])[[1]]

f <- extract_file(lfs_files[[i]], tab_file_name)
print(
    f |>
        read_tab(5, seq(5))
)

# A tibble: 5 × 5
  accdnt acckind accurh1 accurh2 accurh3
   <dbl>   <dbl>   <dbl>   <dbl>   <dbl>
1     -9      -9      -9      -9      -9
2     -9      -9      -9      -9      -9
3     -9      -9      -9      -9      -9
4      2      -9      -9      -9      -9
5     -9      -9      -9      -9      -9
